# Day3_21 · 전공 데이터 분석 프로젝트

건축에너지·콘크리트·실내환경·기상 중 한 경로를 선택해 분석 질문과 결과를 완성합니다.

## 이 노트북에서 확인할 내용

- 전공 질문에 맞는 자료와 방법을 선택할 수 있습니다.
- 표·그림·평가값을 연결해 결과를 설명할 수 있습니다.
- Codex가 제안한 분석을 실행 전에 검토하고 한계를 기록할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-21-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-21-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-21-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-21-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-21-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-21-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-21-07 · 건물 에너지자료 읽기

In [ ]:
building = pd.read_csv(DATA_DIR / 'building_energy.csv')

In [ ]:
building.head()

### D3B-21-08 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete.head()

### D3B-21-09 · 실내 재실자료 읽기

In [ ]:
occupancy = pd.read_csv(DATA_DIR / 'occupancy_detection.csv')

In [ ]:
occupancy.head()

### D3B-21-10 · 울산 기상자료 읽기

In [ ]:
weather = pd.read_csv(DATA_DIR / 'weather_daily.csv')

In [ ]:
weather.head()

### D3B-21-11 · 날짜를 날짜 형식으로 바꾸기

In [ ]:
weather['date'] = pd.to_datetime(weather['date'])

In [ ]:
weather[['date']].head()

### D3B-21-12 · 프로젝트 경로 확인

In [ ]:
project_routes = pd.DataFrame({
    'route': ['building', 'concrete', 'occupancy', 'weather'],
    'question': [
        '건물 외피조건으로 난방 부하를 얼마나 예측할 수 있는가?',
        '배합과 재령으로 압축강도를 얼마나 예측할 수 있는가?',
        '실내 센서값으로 재실 상태를 얼마나 찾을 수 있는가?',
        '2024년 울산 기상에서 계절 패턴과 특이한 날은 무엇인가?',
    ],
    'main_result': ['RMSE와 순열 중요도', 'RMSE와 회귀계수', '혼동행렬과 Recall', '월별 표와 기온 이상값'],
})

In [ ]:
project_routes

아래 네 경로의 작은 예시를 모두 실행한 뒤 한 경로를 선택합니다. 최종 발표에서는 선택한 경로의 질문, 결과표, 그림, 해석과 한계를 사용합니다.

## A. 건축환경 · 난방 부하 예측

### D3B-21-13 · 건물 입력 열 정하기

In [ ]:
building_features = ['relative_compactness', 'surface_area_m2', 'wall_area_m2', 'roof_area_m2', 'overall_height_m', 'orientation', 'glazing_area_ratio', 'glazing_distribution']

In [ ]:
pd.Series(building_features)

### D3B-21-14 · 건물 입력표와 결과 열 만들기

In [ ]:
building_X = building[building_features]
building_y = building['heating_load']

In [ ]:
pd.Series({'input_rows': len(building_X), 'target_rows': len(building_y)})

### D3B-21-15 · 자료 분할과 랜덤포레스트 불러오기

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

### D3B-21-16 · 건물 학습자료와 평가자료 나누기

In [ ]:
building_X_train, building_X_test, building_y_train, building_y_test = train_test_split(building_X, building_y, test_size=0.25, random_state=42)

In [ ]:
pd.Series({'train': len(building_X_train), 'test': len(building_X_test)})

### D3B-21-17 · 건물 랜덤포레스트 학습하기

In [ ]:
building_model = RandomForestRegressor(n_estimators=200, min_samples_leaf=2, random_state=42, n_jobs=-1).fit(building_X_train, building_y_train)

In [ ]:
building_model

### D3B-21-18 · 건물 난방 부하 예측하기

In [ ]:
building_prediction = building_model.predict(building_X_test)

In [ ]:
pd.Series(building_prediction[:5], name='predicted_heating_load')

### D3B-21-19 · 회귀 평가지표 불러오기

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

### D3B-21-20 · 건물 예측 평가표 만들기

In [ ]:
building_metrics = pd.Series({
    'MAE': mean_absolute_error(building_y_test, building_prediction),
    'RMSE': root_mean_squared_error(building_y_test, building_prediction),
    'R2': r2_score(building_y_test, building_prediction),
})

In [ ]:
building_metrics.round(3)

### D3B-21-21 · 건물 실제값과 예측값 그리기

In [ ]:
sns.scatterplot(x=building_y_test, y=building_prediction, alpha=0.6)
limits = [building_y_test.min(), building_y_test.max()]
plt.plot(limits, limits, color='red', linestyle='--')
plt.xlabel('Actual heating load')
plt.ylabel('Predicted heating load')
plt.show()

## B. 건설재료 · 압축강도 회귀

### D3B-21-22 · 콘크리트 입력 열 정하기

In [ ]:
concrete_features = ['cement_kg_m3', 'slag_kg_m3', 'fly_ash_kg_m3', 'water_kg_m3', 'superplasticizer_kg_m3', 'coarse_aggregate_kg_m3', 'fine_aggregate_kg_m3', 'age_days']

In [ ]:
pd.Series(concrete_features)

### D3B-21-23 · 콘크리트 입력표와 결과 열 만들기

In [ ]:
concrete_X = concrete[concrete_features]
concrete_y = concrete['strength_mpa']

In [ ]:
pd.Series({'input_rows': len(concrete_X), 'target_rows': len(concrete_y)})

### D3B-21-24 · 콘크리트 학습자료와 평가자료 나누기

In [ ]:
concrete_X_train, concrete_X_test, concrete_y_train, concrete_y_test = train_test_split(concrete_X, concrete_y, test_size=0.25, random_state=42)

In [ ]:
pd.Series({'train': len(concrete_X_train), 'test': len(concrete_X_test)})

### D3B-21-25 · 선형회귀 불러오기

In [ ]:
from sklearn.linear_model import LinearRegression

### D3B-21-26 · 콘크리트 다중회귀 학습하기

In [ ]:
concrete_model = LinearRegression().fit(concrete_X_train, concrete_y_train)

In [ ]:
concrete_model

### D3B-21-27 · 콘크리트 강도 예측하기

In [ ]:
concrete_prediction = concrete_model.predict(concrete_X_test)

In [ ]:
pd.Series(concrete_prediction[:5], name='predicted_strength_mpa')

### D3B-21-28 · 콘크리트 예측 평가표 만들기

In [ ]:
concrete_metrics = pd.Series({
    'MAE': mean_absolute_error(concrete_y_test, concrete_prediction),
    'RMSE': root_mean_squared_error(concrete_y_test, concrete_prediction),
    'R2': r2_score(concrete_y_test, concrete_prediction),
})

In [ ]:
concrete_metrics.round(3)

### D3B-21-29 · 콘크리트 회귀계수 표 만들기

In [ ]:
concrete_coefficients = pd.DataFrame({'input_column': concrete_features, 'coefficient': concrete_model.coef_}).sort_values('coefficient')

In [ ]:
concrete_coefficients.round(4)

### D3B-21-30 · 콘크리트 실제값과 예측값 그리기

In [ ]:
sns.scatterplot(x=concrete_y_test, y=concrete_prediction, alpha=0.6)
limits = [concrete_y_test.min(), concrete_y_test.max()]
plt.plot(limits, limits, color='red', linestyle='--')
plt.xlabel('Actual strength (MPa)')
plt.ylabel('Predicted strength (MPa)')
plt.show()

## C. 건축설비 · 실내 재실 분류

### D3B-21-31 · 재실 센서 열 정하기

In [ ]:
occupancy_features = ['temperature_c', 'humidity_percent', 'light_lux', 'co2_ppm', 'humidity_ratio']

In [ ]:
pd.Series(occupancy_features)

### D3B-21-32 · 재실 학습 행 선택

In [ ]:
occupancy_train = occupancy.loc[occupancy['source_split'] == 'train']

In [ ]:
occupancy_train.shape

### D3B-21-33 · 재실 평가 행 선택

In [ ]:
occupancy_test = occupancy.loc[occupancy['source_split'] != 'train']

In [ ]:
occupancy_test.shape

### D3B-21-34 · 재실 학습 입력과 결과 만들기

In [ ]:
occupancy_X_train = occupancy_train[occupancy_features]
occupancy_y_train = occupancy_train['occupied']

In [ ]:
pd.Series({'X_rows': len(occupancy_X_train), 'y_rows': len(occupancy_y_train)})

### D3B-21-35 · 재실 평가 입력과 결과 만들기

In [ ]:
occupancy_X_test = occupancy_test[occupancy_features]
occupancy_y_test = occupancy_test['occupied']

In [ ]:
pd.Series({'X_rows': len(occupancy_X_test), 'y_rows': len(occupancy_y_test)})

### D3B-21-36 · 표준화와 로지스틱 회귀 불러오기

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### D3B-21-37 · 재실 센서 변환 규칙 익히기

In [ ]:
occupancy_scaler = StandardScaler().fit(occupancy_X_train)

In [ ]:
occupancy_scaler

### D3B-21-38 · 재실 로지스틱 회귀 학습하기

In [ ]:
occupancy_model = LogisticRegression(max_iter=2000, random_state=42).fit(occupancy_scaler.transform(occupancy_X_train), occupancy_y_train)

In [ ]:
occupancy_model

### D3B-21-39 · 재실 확률 계산하기

In [ ]:
occupancy_probability = occupancy_model.predict_proba(occupancy_scaler.transform(occupancy_X_test))[:, 1]

In [ ]:
pd.Series(occupancy_probability[:5], name='probability_occupied')

### D3B-21-40 · 재실 최종 판단 만들기

In [ ]:
occupancy_prediction = (occupancy_probability >= 0.5).astype(int)

In [ ]:
pd.Series(occupancy_prediction).value_counts().sort_index()

### D3B-21-41 · 분류 평가지표 불러오기

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

### D3B-21-42 · 재실 분류 평가표 만들기

In [ ]:
occupancy_metrics = pd.Series({
    'accuracy': accuracy_score(occupancy_y_test, occupancy_prediction),
    'precision': precision_score(occupancy_y_test, occupancy_prediction),
    'recall': recall_score(occupancy_y_test, occupancy_prediction),
    'F1': f1_score(occupancy_y_test, occupancy_prediction),
})

In [ ]:
occupancy_metrics.round(3)

### D3B-21-43 · 재실 혼동행렬 만들기

In [ ]:
occupancy_confusion = pd.DataFrame(confusion_matrix(occupancy_y_test, occupancy_prediction), index=['actual empty', 'actual occupied'], columns=['predicted empty', 'predicted occupied'])

In [ ]:
occupancy_confusion

### D3B-21-44 · 재실 혼동행렬 그리기

In [ ]:
sns.heatmap(occupancy_confusion, annot=True, fmt='d', cmap='Blues')
plt.show()

## D. 건설환경 · 기상 패턴과 특이한 날

### D3B-21-45 · 기상자료에 월 추가하기

In [ ]:
weather['month'] = weather['date'].dt.month

In [ ]:
weather[['date', 'month']].head()

### D3B-21-46 · 월별 기상 결과표 만들기

In [ ]:
weather_monthly = weather.groupby('month').agg(
    mean_temperature_c=('temperature_mean_c', 'mean'),
    precipitation_total_mm=('precipitation_mm', 'sum'),
    mean_humidity_percent=('humidity_percent', 'mean'),
)

In [ ]:
weather_monthly.round(2)

### D3B-21-47 · 각 행에 월평균기온 붙이기

In [ ]:
weather['month_temperature_mean'] = weather.groupby('month')['temperature_mean_c'].transform('mean')

In [ ]:
weather[['date', 'temperature_mean_c', 'month_temperature_mean']].head()

### D3B-21-48 · 월평균과의 기온 차이 만들기

In [ ]:
weather['temperature_anomaly_c'] = weather['temperature_mean_c'] - weather['month_temperature_mean']

In [ ]:
weather[['date', 'temperature_anomaly_c']].head().round(2)

### D3B-21-49 · 특히 따뜻했던 열흘 찾기

In [ ]:
weather_warmest_anomalies = weather.nlargest(10, 'temperature_anomaly_c')

In [ ]:
weather_warmest_anomalies[['date', 'temperature_mean_c', 'month_temperature_mean', 'temperature_anomaly_c']].round(2)

### D3B-21-50 · 월별 평균기온과 강수량 그리기

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
sns.lineplot(data=weather_monthly.reset_index(), x='month', y='mean_temperature_c', marker='o', ax=axes[0])
sns.barplot(data=weather_monthly.reset_index(), x='month', y='precipitation_total_mm', color='steelblue', ax=axes[1])
axes[0].set_ylabel('Mean temperature (°C)')
axes[1].set_ylabel('Precipitation total (mm)')
plt.show()

## 한 경로 선택하고 발표 카드 만들기

### D3B-21-51 · 프로젝트 경로 선택

In [ ]:
# building, concrete, occupancy, weather 중 하나를 적으세요.
selected_route = 'building'

In [ ]:
selected_route

### D3B-21-52 · 선택한 질문 확인

In [ ]:
selected_question = project_routes.set_index('route').loc[selected_route, 'question']

In [ ]:
selected_question

### D3B-21-53 · 선택한 핵심 결과 확인

In [ ]:
selected_result_name = project_routes.set_index('route').loc[selected_route, 'main_result']

In [ ]:
selected_result_name

### D3B-21-54 · 선택 경로의 결과표 연결

In [ ]:
route_tables = {
    'building': building_metrics,
    'concrete': concrete_metrics,
    'occupancy': occupancy_metrics,
    'weather': weather_monthly,
}
selected_result_table = route_tables[selected_route]

In [ ]:
selected_result_table.round(3)

### D3B-21-55 · 결과 문장 작성

In [ ]:
# 선택한 결과표에서 숫자 하나를 넣어 문장을 완성하세요.
result_sentence = '결과표에서 확인한 숫자와 그 단위를 적습니다.'

In [ ]:
result_sentence

### D3B-21-56 · 그림 해석 문장 작성

In [ ]:
# 선택한 경로의 그림에서 보이는 패턴을 적으세요.
figure_sentence = '그림의 축과 눈에 띄는 패턴을 적습니다.'

In [ ]:
figure_sentence

### D3B-21-57 · 분석 한계 문장 작성

In [ ]:
# 표본, 자료 출처, 인과해석 또는 적용범위의 한계를 적으세요.
limitation_sentence = '이 자료와 방법만으로 판단할 수 없는 내용을 적습니다.'

In [ ]:
limitation_sentence

### D3B-21-58 · 최종 발표 카드 만들기

In [ ]:
final_card = pd.Series({
    '선택 경로': selected_route,
    '질문': selected_question,
    '사용한 결과': selected_result_name,
    '숫자 해석': result_sentence,
    '그림 해석': figure_sentence,
    '한계': limitation_sentence,
})

In [ ]:
final_card

### D3B-21-59 · 완료 확인표 만들기

In [ ]:
completion_check = pd.Series({
    'question_selected': selected_question != '',
    'result_sentence_written': '적습니다' not in result_sentence,
    'figure_sentence_written': '적습니다' not in figure_sentence,
    'limitation_written': '적습니다' not in limitation_sentence,
})

In [ ]:
completion_check

### D3B-21-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''제가 선택한 경로는 selected_route이고 질문은 selected_question입니다. 기존 셀과 기존 변수를 수정하지 마세요. selected_result_table을 보완하는 표 또는 그래프 한 개만 새 셀로 제안해 주세요. 먼저 사용할 열, 행의 의미, 두 축 또는 표 열의 단위를 설명하세요. 마지막에 결과를 과장하지 않았는지 사람이 확인할 질문 세 개를 적어 주세요.'''.strip()

In [ ]:
codex_request

### D3B-21-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['선택한 질문과 직접 연결되는가?', '기존 결과를 덮어쓰지 않는가?', '숫자·단위·자료의 한계를 사람이 설명할 수 있는가?'], name='확인할 내용')

In [ ]:
codex_checklist

## 제출할 내용

- 질문 한 문장
- 결과표 한 개
- 그래프 또는 지도 한 개
- 통계량이나 모델 평가값 한 개
- 결과 해석 두 문장
- 분석의 한계 한 문장
- Codex가 제안한 코드와 사람이 확인한 내용